# Semana 7 · Consultas SQL y pandas — Caso Spotify
**Autor:** Julián Andrés Solano Ledesma · CORHUILA · 2026-B

Tres consultas SQL (WHERE, JOIN, GROUP BY) sobre el modelo normalizado de la Semana 6 y la réplica de la consulta GROUP BY con `pandas.groupby` [1]–[3].

## 1. Carga del modelo normalizado en SQLite

In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect(':memory:')
tablas = ['pais','genero','plan_suscripcion','artista','album','metrica_streaming']
for t in tablas:
    pd.read_csv(f'datos/{t}.csv').to_sql(t, con, index=False)
pd.read_sql("SELECT name, (SELECT COUNT(*) FROM metrica_streaming) AS filas_metrica FROM sqlite_master WHERE type='table'", con)

,name,filas_metrica
0,pais,500
1,genero,500
2,plan_suscripcion,500
3,artista,500
4,album,500
5,metrica_streaming,500


## 2. Q1 · `WHERE` — alto volumen y alto abandono
**Responde:** ¿qué registros de país–álbum superan 150 M de streams en 30 días y, a la vez, un *skip rate* > 30 %? Son los puntos de mayor demanda con peor aceptación del contenido [1].

In [2]:
q1 = '''SELECT metric_id, country_id, streams_30d_m, skip_rate_pct
FROM metrica_streaming
WHERE skip_rate_pct > 30 AND streams_30d_m > 150
ORDER BY skip_rate_pct DESC'''
r1 = pd.read_sql(q1, con); print(len(r1), 'registros'); r1.head(10)

24 registros


,metric_id,country_id,streams_30d_m,skip_rate_pct
0,133,6,163.39,39.14
1,9,1,155.45,38.61
2,163,8,170.26,37.30
3,343,14,192.91,37.23
4,435,17,193.83,37.13
5,82,4,156.24,36.95
6,8,1,165.93,36.21
7,387,16,166.73,36.21
8,176,8,190.06,35.77
9,194,9,175.86,35.73


## 3. Q2 · `JOIN` — top 10 país–artista por streams (30 días)
**Responde:** ¿en qué países y con qué artista/álbum se concentra la mayor demanda reciente? Une cuatro tablas mediante sus claves PK–FK [1].

In [3]:
q2 = '''SELECT p.nombre AS pais, ar.nombre AS artista, al.titulo AS album, m.streams_30d_m
FROM metrica_streaming m
JOIN pais p ON p.country_id = m.country_id
JOIN album al ON al.album_id = m.album_id
JOIN artista ar ON ar.artist_id = al.artist_id
ORDER BY m.streams_30d_m DESC LIMIT 10'''
pd.read_sql(q2, con)

,pais,artista,album,streams_30d_m
0,Italy,Taylor Swift,1989 (Taylor's Version),200.00
1,Brazil,Ed Sheeran,Autumn Variations,199.91
2,Argentina,Ariana Grande,Eternal Sunshine,198.06
3,India,BTS,Proof,197.84
4,South Korea,SZA,SOS,197.83
5,France,Bad Bunny,Nadie Sabe Lo Que Va a Pasar Mañana,197.08
6,United States,BLACKPINK,BORN PINK,196.16
7,France,Drake,For All The Dogs,196.15
8,Brazil,Ariana Grande,Eternal Sunshine,194.95
9,Argentina,The Weeknd,After Hours,194.82


## 4. Q3 · `GROUP BY` — capacidad y abandono por país
**Responde:** ¿qué países concentran más streams en 30 días y cuál es su abandono promedio? Agrega con `COUNT`, `SUM` y `AVG` [1].

In [4]:
q3 = '''SELECT country_id, COUNT(*) AS n_registros,
       ROUND(SUM(streams_30d_m),2) AS streams_30d_total_m,
       ROUND(AVG(skip_rate_pct),2) AS skip_rate_prom_pct
FROM metrica_streaming GROUP BY country_id ORDER BY skip_rate_prom_pct DESC'''
r3 = pd.read_sql(q3, con); r3.head(10)

,country_id,n_registros,streams_30d_total_m,skip_rate_prom_pct
0,8,25,2197.80,23.03
1,10,23,2471.66,22.76
2,1,27,2834.22,22.62
3,9,34,3227.25,22.37
4,14,32,3024.99,22.07
5,13,31,2886.25,21.90
6,17,30,3023.20,21.79
7,15,31,3171.34,20.97
8,3,30,3063.11,20.59
9,4,25,2589.97,20.52


## 5. Equivalente en pandas — `groupby` de Q3
`groupby` aplica el patrón *split–apply–combine* [2], [3]: divide por `country_id`, agrega y recombina.

In [5]:
m = pd.read_csv('datos/metrica_streaming.csv')
p3 = (m.groupby('country_id')
        .agg(n_registros=('metric_id','count'),
             streams_30d_total_m=('streams_30d_m','sum'),
             skip_rate_prom_pct=('skip_rate_pct','mean'))
        .round(2)
        .sort_values('skip_rate_prom_pct', ascending=False)
        .reset_index())
p3.head(10)

,country_id,n_registros,streams_30d_total_m,skip_rate_prom_pct
0,8,25,2197.80,23.03
1,10,23,2471.66,22.76
2,1,27,2834.22,22.62
3,9,34,3227.25,22.37
4,14,32,3024.99,22.07
5,13,31,2886.25,21.90
6,17,30,3023.20,21.79
7,15,31,3171.34,20.97
8,3,30,3063.11,20.59
9,4,25,2589.97,20.52


## 6. Verificación de equivalencia SQL ↔ pandas

In [6]:
a = r3.sort_values('country_id').reset_index(drop=True)
b = p3.sort_values('country_id').reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_dtype=False, atol=0.011)
print('OK: SQL (Q3) y pandas producen el mismo resultado en', len(a), 'países')

OK: SQL (Q3) y pandas producen el mismo resultado en 20 países


## 7. Interpretación con nombres de país

In [7]:
out = p3.merge(pd.read_csv('datos/pais.csv')[['country_id','nombre']], on='country_id')
out[['nombre','n_registros','streams_30d_total_m','skip_rate_prom_pct']].head(5)

,nombre,n_registros,streams_30d_total_m,skip_rate_prom_pct
0,Indonesia,25,2197.80,23.03
1,Japan,23,2471.66,22.76
2,Argentina,27,2834.22,22.62
3,Italy,34,3227.25,22.37
4,South Africa,32,3024.99,22.07


## Referencias
[1] A. Watt y N. Eng, *Database Design*, 2.ª ed., BCcampus, 2014. https://opentextbc.ca/dbdesign01/chapter/chapter-sql-dml/

[2] W. McKinney, «Data structures for statistical computing in Python», *Proc. 9th Python in Science Conf.*, 2010, pp. 56–61.

[3] H. Wickham, «The split-apply-combine strategy for data analysis», *J. Stat. Softw.*, vol. 40, n.º 1, pp. 1–29, 2011.